# 10 · Final Random Forest: gold labels + real tree height

**Run this in Google Colab after notebook 09** (it needs `data/features/canopy_features.parquet`) and after you
uploaded `adjudication.csv` to `data/labels`. It takes about 30–40 minutes.

Changes from notebook 08:
1. **Gold labels** (`LABEL_MODE = 'gold'`): where you and the AI agree on good vs not good, your label is kept;
   where you disagree, your **blind re-check** (`adjudication.csv`) decides. `'ai'` or `'hand'` still work.
2. **Real tree height** from the Meta 1 m canopy height map (notebook 09): 4 new shade features.
3. The accuracy table now compares, on the same roofs: the model **without** and **with** the tree-height features,
   and the model trained and tested **on real buildings only** ("not a building" removed).

What it does *(Module 3: classification on spatial big data; Module 4: environment and economics)*:
reads the labels, checks accuracy honestly (*leave-one-district-out*: train on 13 districts, test on the 14th),
trains the final model, shows which features matter, **predicts all ~13 million roofs**, gives the
**realistic solar potential per district**, and runs **FP-Growth** on the suitable roofs.

"Not a building" (2) counts as **not suitable**. Empty labels are skipped.
Everything printed is also saved to `MyDrive/rooftop_solar/logs/10_log_<mode>.txt`.

**How to run:** Runtime → Run all. Allow Google Drive access.

In [ ]:
# Step 1: connect Google Drive, set folders, copy the inputs to the fast Colab disk
from google.colab import drive
drive.mount('/content/drive')

import os, shutil, time

PROJECT = '/content/drive/MyDrive/rooftop_solar'
FEATURES = f'{PROJECT}/data/features'
LABELS = f'{PROJECT}/data/labels'
RESULTS = f'{PROJECT}/results'
LOGS = f'{PROJECT}/logs'
LOCAL = '/content/work'
for folder in (RESULTS, LOGS, LOCAL):
    os.makedirs(folder, exist_ok=True)

LABEL_MODE = 'gold'   # 'gold' = agreed + blind-checked (best), 'ai' = AI labels, 'hand' = your first labels
LOG_FILE = f'{LOGS}/10_log_{LABEL_MODE}.txt'
open(LOG_FILE, 'w').close()


def log(*parts):
    """Print, and also save to Google Drive so results survive if the tab closes."""
    text = ' '.join(str(p) for p in parts)
    print(text)
    with open(LOG_FILE, 'a') as f:
        f.write(text + '\n')


def log_table(pdf, csv_name=None):
    """Log a small pandas table; optionally save it as a CSV in results/."""
    log(pdf.to_string(index=False))
    if csv_name:
        pdf.to_csv(f'{RESULTS}/{csv_name}', index=False)


T_START = time.time()
shutil.copytree(f'{FEATURES}/roof_features_v2', f'{LOCAL}/roof_features_v2', dirs_exist_ok=True)
shutil.copy(f'{FEATURES}/hex_clusters.parquet', f'{LOCAL}/hex_clusters.parquet')
shutil.copy(f'{FEATURES}/canopy_features.parquet', f'{LOCAL}/canopy_features.parquet')
for name in ('roofs_to_label.gpkg', 'roofs_to_label_ai.gpkg', 'adjudication.csv'):
    if LABEL_MODE == 'gold' or name != 'adjudication.csv':
        shutil.copy(f'{LABELS}/{name}', f'{LOCAL}/{name}')
log(f'Inputs copied to {LOCAL} in {time.time() - T_START:.0f} s')

In [ ]:
# Step 2: install Java 17, PySpark 4.1 and DuckDB (to read the GeoPackage)
import subprocess


def java_version():
    java_bin = os.path.join(os.environ['JAVA_HOME'], 'bin', 'java') if 'JAVA_HOME' in os.environ else 'java'
    out = subprocess.run(f'{java_bin} -version 2>&1', shell=True, capture_output=True, text=True).stdout
    return out.splitlines()[0] if out else 'no java'


if not any(f'version "{v}' in java_version() for v in ('17', '21')):
    !apt-get -qq update > /dev/null && apt-get -qq install -y openjdk-17-jdk-headless > /dev/null
    os.environ['JAVA_HOME'] = '/usr/lib/jvm/java-17-openjdk-amd64'
!pip install -q pyspark==4.1.3 duckdb==2.0.0
log('Java:', java_version())

In [ ]:
# Step 3: read the labels (yours, the AI's, and your blind re-checks)
import duckdb
import numpy as np
import pandas as pd

dk = duckdb.connect()
dk.sql("INSTALL spatial; LOAD spatial;")
labels = dk.execute(f"SELECT id, district, label FROM ST_Read('{LOCAL}/roofs_to_label.gpkg')").df()
ai = dk.execute(f"SELECT id, ai_label FROM ST_Read('{LOCAL}/roofs_to_label_ai.gpkg')").df()
dk.close()
labels = labels.merge(ai, on='id', how='left')
for col in ('label', 'ai_label'):
    labels[col] = pd.to_numeric(labels[col], errors='coerce')

if LABEL_MODE == 'gold':
    adj = pd.read_csv(f'{LOCAL}/adjudication.csv', dtype={'id': str})
    both = labels['label'].notna() & labels['ai_label'].notna()
    agree = both & ((labels['label'] == 1) == (labels['ai_label'] == 1))
    verdict = labels['id'].map(dict(zip(adj['id'], adj['verdict'])))
    labels['label'] = np.where(agree, labels['label'], verdict)   # disagreement: the blind re-check decides
    n_dis, n_chk = int((both & ~agree).sum()), int((both & ~agree & verdict.notna()).sum())
    log(f'Gold labels: {int(agree.sum())} roofs where you and the AI agree + {n_chk} of {n_dis} disagreements blind-checked')
    if n_chk < n_dis:
        log(f'  WARNING: {n_dis - n_chk} disagreements are not blind-checked yet, so they are left out')
elif LABEL_MODE == 'ai':
    labels['label'] = labels['ai_label']
bad = labels[labels['label'].notna() & ~labels['label'].isin([0, 1, 2])]
if len(bad):
    raise SystemExit(f'Some labels are not 0, 1 or 2 (rows: {bad.index.tolist()[:10]}). Fix them in QGIS first.')

summary = (labels['label'].map({1: '1 = good', 0: '0 = bad', 2: '2 = not a building'}).fillna('empty (skipped)')
           .value_counts().rename_axis('label').reset_index(name='roofs'))
log(f'\nLABELS ({LABEL_MODE})')
log_table(summary, 'label_summary.csv')

labelled = labels[labels['label'].notna()].copy()
labelled['y'] = (labelled['label'] == 1).astype(float)            # suitable = good; bad and not-a-building = 0
n_good, n_not = int(labelled['y'].sum()), int((labelled['y'] == 0).sum())
if len(labelled) < 100 or min(n_good, n_not) < 15:
    raise SystemExit(f'Only {len(labelled)} labelled roofs ({n_good} good, {n_not} not good). '
                     'Label at least 100, with 15+ of each kind, then run again.')
log(f'Training labels: {len(labelled)} roofs ({n_good} good, {n_not} not good) from {labelled.district.nunique()} districts')

In [ ]:
# Step 4: start Spark and build the feature table for all roofs
import psutil
from pyspark.sql import SparkSession, functions as F

CORES = os.cpu_count()
MEM_GB = max(2, int(psutil.virtual_memory().total / 1e9 * 0.6))
spark = (SparkSession.builder.master('local[*]').appName('rooftop_rf')
         .config('spark.driver.memory', f'{MEM_GB}g')
         .config('spark.sql.shuffle.partitions', str(CORES * 4))
         .getOrCreate())
log(f'\nSpark {spark.version} | CPU cores: {CORES} | Spark memory: {MEM_GB} GB')

spark.read.parquet(f'{LOCAL}/roof_features_v2').createOrReplaceTempView('roofs')
spark.read.parquet(f'{LOCAL}/hex_clusters.parquet').select('h3_9', 'cluster').createOrReplaceTempView('hexc')
spark.read.parquet(f'{LOCAL}/canopy_features.parquet').createOrReplaceTempView('canopy')

FEATS_OLD = ['log_area', 'compactness', 'tree_28m', 'tree_140m', 'tree_300m', 'seen_built_up',
             'log_crowding', 'ai_confidence', 'low_conf', 'source_code', 'area_type']
CANOPY = ['chm_roof_max', 'chm_max_15m', 'tall_pct_15m', 'tall_pct_30m']   # real tree height (notebook 09)
FEATS = FEATS_OLD + CANOPY
features = spark.sql("""
    SELECT /*+ BROADCAST(h) */
           r.id, r.district, r.area_m2, r.kwp, r.kwh_year, r.centre_lon, r.centre_lat, r.h3_9,
           r.roof_seen_as, r.tree_pct_300m, r.bldgs_in_hex, r.compactness AS compactness_raw, r.low_confidence,
           log1p(r.area_m2)                                         AS log_area,
           coalesce(r.compactness, 0.7)                             AS compactness,
           coalesce(r.tree_pct, 100D)                               AS tree_28m,
           coalesce(r.tree_pct_140m, 100D)                          AS tree_140m,
           coalesce(r.tree_pct_300m, 100D)                          AS tree_300m,
           CAST(coalesce(r.roof_seen_as = 'built-up', false) AS DOUBLE) AS seen_built_up,
           log1p(r.bldgs_in_hex)                                    AS log_crowding,
           r.pvout_kwh_per_kwp                                      AS pvout,
           coalesce(r.confidence, 1D)                               AS ai_confidence,
           CAST(r.low_confidence AS DOUBLE)                         AS low_conf,
           CASE r.source WHEN 'Google Open Buildings' THEN 0D WHEN 'OpenStreetMap' THEN 1D ELSE 2D END AS source_code,
           CAST(coalesce(h.cluster, 0) AS DOUBLE)                   AS area_type,
           CAST(coalesce(c.chm_roof_max, 0) AS DOUBLE)              AS chm_roof_max,
           CAST(coalesce(c.chm_max_15m, 0) AS DOUBLE)               AS chm_max_15m,
           CAST(coalesce(c.tall_pct_15m, 0) AS DOUBLE)              AS tall_pct_15m,
           CAST(coalesce(c.tall_pct_30m, 0) AS DOUBLE)              AS tall_pct_30m,
           c.id IS NULL                                             AS no_canopy
    FROM roofs r LEFT JOIN hexc h ON r.h3_9 = h.h3_9
                 LEFT JOIN canopy c ON r.id = c.id
""")
features.createOrReplaceTempView('features')

lab = spark.createDataFrame(labelled[['id', 'label', 'y']])
train = (features.join(F.broadcast(lab), 'id').cache())
N_TRAIN = train.count()
log(f'Labelled roofs matched to features: {N_TRAIN} of {len(labelled)} '
    f'({train.filter("no_canopy").count()} of them without canopy data)')

# Balance the two classes so the model does not just predict the common one
w_good = N_TRAIN / (2 * max(train.filter('y = 1').count(), 1))
w_not = N_TRAIN / (2 * max(train.filter('y = 0').count(), 1))
train = train.withColumn('w', F.when(F.col('y') == 1, w_good).otherwise(w_not)).cache()

In [ ]:
# Step 5: the Random Forest pipeline, and an honest accuracy check
from pyspark.ml import Pipeline
from pyspark.ml.classification import RandomForestClassifier
from pyspark.ml.feature import VectorAssembler, VectorIndexer
from pyspark.ml.functions import vector_to_array


def make_pipeline(feats=FEATS):
    return Pipeline(stages=[
        VectorAssembler(inputCols=feats, outputCol='raw'),
        VectorIndexer(inputCol='raw', outputCol='features', maxCategories=3, handleInvalid='keep'),
        RandomForestClassifier(labelCol='y', featuresCol='features', weightCol='w', numTrees=200, maxDepth=6,
                               minInstancesPerNode=3, featureSubsetStrategy='sqrt', seed=42),
    ])


def scores(y, p, threshold=0.5):
    """Accuracy, precision, recall, F1 (for 'good'), balanced accuracy and ROC AUC."""
    y = np.asarray(y, dtype=int)
    p = np.asarray(p, dtype=float)
    pred = (p >= threshold).astype(int)
    tp, tn = int(((pred == 1) & (y == 1)).sum()), int(((pred == 0) & (y == 0)).sum())
    fp, fn = int(((pred == 1) & (y == 0)).sum()), int(((pred == 0) & (y == 1)).sum())
    prec = tp / (tp + fp) if tp + fp else 0.0
    rec = tp / (tp + fn) if tp + fn else 0.0
    spec = tn / (tn + fp) if tn + fp else 0.0
    ranks = pd.Series(p).rank().to_numpy()                     # AUC = chance a good roof scores above a bad one
    n1, n0 = int(y.sum()), int(len(y) - y.sum())
    auc = (ranks[y == 1].sum() - n1 * (n1 + 1) / 2) / (n1 * n0) if n1 and n0 else float('nan')
    return {'accuracy': round((tp + tn) / len(y), 3), 'precision': round(prec, 3), 'recall': round(rec, 3),
            'f1': round(2 * prec * rec / (prec + rec), 3) if prec + rec else 0.0,
            'balanced_accuracy': round((rec + spec) / 2, 3), 'roc_auc': round(auc, 3),
            'tp': tp, 'fp': fp, 'tn': tn, 'fn': fn}


def predict_proba(model, df):
    return (model.transform(df).withColumn('p', vector_to_array('probability')[1])
            .select('id', 'district', 'y', 'p').toPandas())


t = time.time()
# (a) leave-one-district-out: the model never sees the test district (honest for new places)
def leave_one_district_out(data, feats):
    out = []
    for d in sorted(r[0] for r in data.select('district').distinct().collect()):
        tr, te = data.filter(F.col('district') != d), data.filter(F.col('district') == d)
        if tr.select('y').distinct().count() < 2:
            continue
        out.append(predict_proba(make_pipeline(feats).fit(tr), te))
    return pd.concat(out)


lodo = leave_one_district_out(train, FEATS)
lodo_old = leave_one_district_out(train, FEATS_OLD)                     # same roofs, without tree height
real = train.filter('label != 2')                                        # "not a building" removed
lodo_real = leave_one_district_out(real, FEATS)

# (b) random 5-fold: easier, usually more optimistic (test roofs can sit next to training roofs)
folded = train.withColumn('fold', F.pmod(F.hash('id'), F.lit(5)))
rand = pd.concat([predict_proba(make_pipeline().fit(folded.filter(f'fold != {k}')), folded.filter(f'fold = {k}'))
                  for k in range(5)])

# (c) a simple rule to beat: "suitable if the roof is at least 40 m2 and the satellite sees it as built-up"
rule = train.select('y', (F.col('area_m2') >= 40) & (F.col('seen_built_up') == 1)).toPandas()

metrics = pd.DataFrame([
    {'check': 'RF without tree height - leave one district out', **scores(lodo_old['y'], lodo_old['p'])},
    {'check': 'RF with tree height - leave one district out', **scores(lodo['y'], lodo['p'])},
    {'check': 'RF with tree height - random 5-fold', **scores(rand['y'], rand['p'])},
    {'check': 'RF with tree height - real buildings only (no label 2)', **scores(lodo_real['y'], lodo_real['p'])},
    {'check': 'Simple rule (>= 40 m2 and seen built-up)', **scores(rule.iloc[:, 0], rule.iloc[:, 1].astype(float))},
    {'check': 'Always say "not good" (baseline)', **scores(rule.iloc[:, 0], np.zeros(len(rule)))},
    {'check': 'Always say "good" (baseline)', **scores(rule.iloc[:, 0], np.ones(len(rule)))},
])
log(f'\nACCURACY CHECKS ({(time.time() - t) / 60:.1f} min)')
log_table(metrics, 'rf_accuracy.csv')

per_district = (lodo.assign(correct=((lodo['p'] >= 0.5).astype(int) == lodo['y']))
                .groupby('district').agg(labelled=('y', 'size'), accuracy=('correct', 'mean'))
                .round({'accuracy': 2}).reset_index())
log('\nLEAVE-ONE-DISTRICT-OUT ACCURACY BY DISTRICT')
log_table(per_district, 'rf_accuracy_by_district.csv')

In [ ]:
# Step 6: final model on all labels + which features matter most
t = time.time()
final = make_pipeline().fit(train)
imp = final.stages[-1].featureImportances.toArray()
importance = (pd.DataFrame({'feature': FEATS, 'importance': np.round(imp, 3)})
              .sort_values('importance', ascending=False))
log('\nFEATURE IMPORTANCE (share of the model\'s decisions, adds up to 1)')
log_table(importance, 'rf_feature_importance.csv')

# What "not a building" (2) tells us about the AI shapes
nb = train.select('label', 'low_confidence').toPandas()
nb_table = (nb.assign(ai_shape=np.where(nb['low_confidence'], 'low confidence', 'normal'),
                      not_a_building=(nb['label'] == 2))
            .groupby('ai_shape')['not_a_building'].agg(['count', 'sum']).reset_index()
            .rename(columns={'count': 'labelled', 'sum': 'not_a_building'}))
nb_table['pct_not_a_building'] = (100 * nb_table['not_a_building'] / nb_table['labelled']).round(1)
log('\nFAKE BUILDINGS FOUND WHILE LABELLING')
log_table(nb_table, 'not_a_building_by_ai_confidence.csv')
log(f'Final model trained in {time.time() - t:.0f} s')

In [ ]:
# Step 7: predict for all roofs and save
t = time.time()
scored = (final.transform(features)
          .withColumn('p_suitable', F.round(vector_to_array('probability')[1], 3))
          .withColumn('suitable', F.col('p_suitable') >= 0.5)
          .select('id', 'district', 'p_suitable', 'suitable', 'area_type', 'area_m2', 'kwp', 'kwh_year',
                  'centre_lon', 'centre_lat', 'h3_9', 'roof_seen_as', 'tree_pct_300m', 'bldgs_in_hex', *CANOPY,
                  F.col('compactness_raw').alias('compactness'), 'low_confidence'))
shutil.rmtree(f'{LOCAL}/roof_predictions', ignore_errors=True)
scored.write.partitionBy('district').mode('overwrite').parquet(f'{LOCAL}/roof_predictions')
log(f'\nPredicted all roofs in {(time.time() - t) / 60:.1f} min')

pred = spark.read.parquet(f'{LOCAL}/roof_predictions').cache()
pred.createOrReplaceTempView('pred')
district_pot = spark.sql("""
    SELECT district,
           count(*)                                                   AS roofs,
           sum(CAST(suitable AS INT))                                 AS suitable_roofs,
           round(100 * avg(CAST(suitable AS DOUBLE)), 1)              AS pct_suitable,
           round(sum(kwp) / 1e6, 2)                                   AS all_roofs_gw,
           round(sum(CASE WHEN suitable THEN kwp END) / 1e6, 2)       AS suitable_gw,
           round(sum(CASE WHEN suitable THEN kwh_year END) / 1e9, 2)  AS suitable_twh_per_year
    FROM pred GROUP BY district ORDER BY suitable_twh_per_year DESC
""").toPandas()
total = pd.DataFrame([{'district': 'KERALA', 'roofs': district_pot.roofs.sum(),
                       'suitable_roofs': district_pot.suitable_roofs.sum(),
                       'pct_suitable': round(100 * district_pot.suitable_roofs.sum() / district_pot.roofs.sum(), 1),
                       'all_roofs_gw': round(district_pot.all_roofs_gw.sum(), 2),
                       'suitable_gw': round(district_pot.suitable_gw.sum(), 2),
                       'suitable_twh_per_year': round(district_pot.suitable_twh_per_year.sum(), 2)}])
log('\nREALISTIC SOLAR POTENTIAL (roofs the model calls suitable)')
log_table(pd.concat([district_pot, total]), 'district_realistic_potential.csv')

log('\nSUITABLE SHARE BY AREA TYPE (0 = homestead villages, 1 = towns, 2 = sparse open land)')
log_table(spark.sql("""
    SELECT CAST(area_type AS INT) AS area_type, count(*) AS roofs,
           round(100 * avg(CAST(suitable AS DOUBLE)), 1) AS pct_suitable,
           round(sum(CASE WHEN suitable THEN kwp END) / 1e6, 2) AS suitable_gw
    FROM pred GROUP BY 1 ORDER BY 1
""").toPandas(), 'suitable_by_area_type.csv')

In [ ]:
# Step 8: FP-Growth again - what do the roofs the MODEL calls suitable have in common? (describes the model, not new facts)
from pyspark.ml.fpm import FPGrowth

t = time.time()
baskets = spark.sql("""
    SELECT filter(array(
        CASE WHEN area_m2 < 50 THEN 'roof=small (<50 m2)' WHEN area_m2 < 120 THEN 'roof=medium (50-120 m2)'
             WHEN area_m2 < 500 THEN 'roof=large (120-500 m2)' ELSE 'roof=very large (>500 m2)' END,
        'seen as=' || coalesce(roof_seen_as, 'unknown'),
        CASE WHEN tree_pct_300m < 70 THEN 'trees 300m=low (<70%)' WHEN tree_pct_300m < 95 THEN 'trees 300m=medium (70-95%)'
             ELSE 'trees 300m=high (>=95%)' END,
        CASE WHEN bldgs_in_hex < 50 THEN 'area=village (<50 bldgs/hex)' WHEN bldgs_in_hex < 150 THEN 'area=suburb (50-150)'
             ELSE 'area=town (>=150)' END,
        CASE WHEN suitable THEN 'SUITABLE=yes' ELSE 'SUITABLE=no' END
    ), x -> x IS NOT NULL) AS items
    FROM pred
""")
fp = FPGrowth(itemsCol='items', minSupport=0.02, minConfidence=0.5).fit(baskets)
rules = fp.associationRules.filter(F.array_contains('consequent', 'SUITABLE=yes')).toPandas()
rules = (rules.assign(n_items=rules['antecedent'].apply(len))
         .sort_values(['n_items', 'confidence'], ascending=[True, False]))
kept, rows = [], []
for row in rules.itertuples():                       # drop longer copies of a shorter rule
    a = set(row.antecedent)
    if any(ka <= a and row.confidence - kc < 0.02 for ka, kc in kept):
        continue
    kept.append((a, row.confidence))
    rows.append(row.Index)
rules = rules.loc[rows]
out = pd.DataFrame({'if_roof_has': rules['antecedent'].apply(' + '.join),
                    'then': 'SUITABLE=yes',
                    'support': rules['support'].round(3), 'confidence': rules['confidence'].round(3),
                    'lift': rules['lift'].round(2)}).sort_values('lift', ascending=False)
log(f'\nRULES FOR SUITABLE ROOFS ({len(out)} rules, {(time.time() - t) / 60:.1f} min)')
log_table(out.head(15), 'fpgrowth_suitable_rules.csv')

In [ ]:
# Step 9: copy the predictions to Google Drive
t = time.time()
shutil.rmtree(f'{FEATURES}/roof_predictions_{LABEL_MODE}', ignore_errors=True)
shutil.copytree(f'{LOCAL}/roof_predictions', f'{FEATURES}/roof_predictions_{LABEL_MODE}')
log(f'\nSaved {FEATURES}/roof_predictions_{LABEL_MODE} in {time.time() - t:.0f} s')
log(f'Total time {(time.time() - T_START) / 60:.1f} min')
log('ALL DONE')

## Done

Send Claude the text of `MyDrive/rooftop_solar/logs/10_log_gold.txt`.